In [ ]:
import os
import re
import sys
import json
import pickle
import contextlib
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict

from lambeq.backend.grammar import Diagram, Id
from lambeq import AtomicType

In [ ]:
d, dv, dt = [], [], []
with open ("Dataset/Raw/cnn_dailymail/train.jsonl") as f:
  for line in f:
    d.append(line)
with open ("Dataset/Raw/cnn_dailymail/val.jsonl") as f:
  for line in f:
    dv.append(line)
with open ("Dataset/Raw/cnn_dailymail/test.jsonl") as f:
  for line in f:
    dt.append(line)


print(len(d), len(dv), len(dt))

In [ ]:
train_save = []
with open ("Dataset/Encoded/train/ds/train_550.pkl", "rb") as f:
# with open ("saves/test/ds/test_80.pkl", "rb") as f:
# with open ("saves/val/ds/val_90.pkl", "rb") as f:
  # loaded_data = pickle.load(f)
  train_save = pickle.load(f)
  #train_save = loaded_data['dste']
  print(list(train_save.keys()))
  print(type(train_save['dste']))
  print(len(train_save['dste']))
  print(list(train_save['dste'][0].keys()))

In [ ]:
train_save2 = []
# with open ("Dataset/Encoded/test/ds/test_0_90.pkl", "rb") as f:
# with open ("Dataset/Encoded/val/ds/val_.pkl", "rb") as f:
with open ("Dataset/Encoded/train/ds/train_100_700.pkl", "rb") as f:
  train_save2 = pickle.load(f)
  print(list(train_save2.keys()))
  print(type(train_save2['dste']))
  print(len(train_save2['dste']))
  print(list(train_save2['dste'][0].keys()))

In [ ]:
# for item in train_save['dste'][:5]:
#   # print(len(item['circuits']), len(item['labels']), len(item['original_text_sentences']))
#   print(item['original_text_sentences'])

# print()
# for item in train_save2['dste'][:5]:
#   # print(len(item['circuits']), len(item['labels']), len(item['original_text_sentences']))
#   print(item['original_text_sentences'])

# for i in range(80):
#   print(train_save['dste'][i]['original_text_sentences'] == train_save2['dste'][i]['original_text_sentences'], train_save['dste'][i]['circuits'] == train_save2['dste'][i]['circuits'], train_save['dste'][i]['labels'] == train_save2['dste'][i]['labels'])


list1 = train_save['dste']
list2 = train_save2['dste']
count = 0
for i, dict1 in enumerate(list1):
    for j, dict2 in enumerate(list2):

        # Extract the values for comparison
        sentences_match = dict1.get('original_text_sentences') == dict2.get('original_text_sentences')
        circuits_match = dict1.get('circuits') == dict2.get('circuits')
        labels_match = dict1.get('labels') == dict2.get('labels')

        if (sentences_match and circuits_match and labels_match):
          count += 1
          # print(f'1.{i} ==', f'2.{j}')

print(count)

In [ ]:
print("veikas, pasaul@")

In [ ]:
##############################################################
##############################################################

#################################################
####             Data processing             ####
#################################################


def flatten(encoded: List[Dict]) -> Tuple[List, List]:
    circuits, labels = [], []
    for dct in encoded:
        circuits.extend(dct["circuits"])
        labels.extend(dct["labels"])
    return circuits, labels


def load_with_pikle(file_path: str, amount: int = None):
    with open(file_path, "rb") as file:
        loaded_data = pickle.load(file)

    dste = loaded_data["dste"]
    length = 0
    for d in dste:
        length += len(d["circuits"])

    print(f"Successfully Loaded {length} circuits!")
    return dste


def load_encoded_data(file_path: str, amount: int = None):
    with open(file_path, "rb") as file:
        loaded_data = pickle.load(file)

    return loaded_data

In [ ]:
loaded_ds = load_encoded_data(file_path="Dataset/Encoded/cnn_dailymail/cnn_dailymail_train_0_9.pkl",amount=10)

In [18]:
print(len(loaded_ds))

10


In [30]:
print(get_deep_type(loaded_ds))

loaded_circuits, loaded_labels = flatten(loaded_ds)
print(get_deep_type(loaded_labels))

List[Dict[str, List[Diagram] | List[List[int]] | List[str]]]
List[List[int]]


In [ ]:
#################################################
####         Encoded data analysation        ####
#################################################

def get_deep_type(obj):
    if isinstance(obj, list):
        # We look at the unique types inside the list to keep it readable
        inner_types = {get_deep_type(item) for item in obj}
        return f"List[{' | '.join(sorted(inner_types))}]"

    elif isinstance(obj, dict):
        # We summarize the types of all keys and all values
        key_types = {get_deep_type(k) for k in obj.keys()}
        val_types = {get_deep_type(v) for v in obj.values()}
        return f"Dict[{' | '.join(sorted(key_types))}, {' | '.join(sorted(val_types))}]"

    else:
        # Return the class name (e.g., 'Diagram' or 'str')
        return type(obj).__name__


def find_mismatches(data_a, data_b):
    # 1. Check if the outer lists are even the same length
    if len(data_a) != len(data_b):
        print(
            f"❌ [CRITICAL] Outer List Length Mismatch: List A={len(data_a)}, List B={len(data_b)}"
        )

    # Iterate through the top-level list
    for i, (dict_a, dict_b) in enumerate(zip(data_a, data_b)):
        # Check if the keys in the dictionaries match
        keys_a = set(dict_a.keys())
        keys_b = set(dict_b.keys())

        if keys_a != keys_b:
            print(f"❌ [Index {i}] Key Mismatch:")
            print(f"   Keys only in A: {keys_a - keys_b}")
            print(f"   Keys only in B: {keys_b - keys_a}")
            continue  # Skip to next list item if keys don't match

        # Check values for each key
        for key in keys_a:
            list_a = dict_a[key]
            list_b = dict_b[key]

            # 2. Check lengths of the lists inside the dictionary
            if len(list_a) != len(list_b):
                print(f"❌ [Index {i}][Key: '{key}'] Inner List Length Mismatch:")
                print(f"   Length A: {len(list_a)}")
                print(f"   Length B: {len(list_b)}")

            # 3. Check individual elements inside those lists
            # This handles List[Diagram], List[List[int]], and List[str]
            for j, (val_a, val_b) in enumerate(zip(list_a, list_b)):
                if val_a != val_b:
                    print(
                        f"❌ [Index {i}][Key: '{key}'][Inner Index {j}] Content Mismatch!"
                    )
                    print(f"   Type A: {type(val_a).__name__}")
                    print(f"   Type B: {type(val_b).__name__}")

                    # If they are small (like List[int] or str), print the actual value
                    if not hasattr(
                        val_a, "draw"
                    ):  # Don't print full Diagrams, they are too big
                        print(f"   Value A: {val_a}")
                        print(f"   Value B: {val_b}")
                    else:
                        print(
                            f"   (Diagram content differs - possibly different boxes or wires)"
                        )

In [ ]:
print(loaded_data[7]["circuits"][11] == encoded_data[7]["circuits"][11])
# print(loaded_data[7]["labels"][11] == encoded_data[7]["labels"][11])
# print(loaded_data[7]["original_text_sentences"][11] == encoded_data[7]["original_text_sentences"][11])

print(len(loaded_data[7]["circuits"][11]), len(encoded_data[7]["circuits"][11]))

k = 0
for i in range(316):
    if loaded_data[7]["circuits"][11][i] != encoded_data[7]["circuits"][11][i]:
        k+=1
        print(k)
        print(loaded_data[7]["circuits"][11][i])
        print(encoded_data[7]["circuits"][11][i])
# print(type(loaded_data[7]["circuits"][11]))

In [ ]:
print(loaded_data[7]["circuits"][11][1])

In [ ]:
print(loaded_data == encoded_data)

In [ ]:
for i, e in enumerate(errors):
    print(f"{i}: {e}")